In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os 
import seaborn as sns
import cv2
import random
import datetime

In [2]:
from keras.models import Sequential, Model, load_model
from keras.layers import Dense,Dropout,Flatten,Conv2D,MaxPooling2D,Input,Activation,GlobalAveragePooling2D, BatchNormalization,Reshape
from keras.optimizers import Adam, RMSprop
from keras.layers import LeakyReLU,Conv2DTranspose
from keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
# from keras.utils import plot_model
from keras.datasets.cifar10 import load_data

In [3]:
def define_discriminator(in_shape=(32,32,3)):
    model = Sequential()
    model.add(Conv2D(64, (3,3), padding='same', input_shape=in_shape))
    model.add(LeakyReLU(alpha=0.2))
    
    model.add(Conv2D(128, (3,3), strides=(2,2), padding='same'))
    model.add(LeakyReLU(alpha=0.2))

    model.add(Conv2D(128, (3,3), strides=(2,2), padding='same'))
    model.add(LeakyReLU(alpha=0.2))

    model.add(Conv2D(256, (3,3), strides=(2,2), padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    # classifier
    model.add(Flatten())
    model.add(Dropout(0.4))
    model.add(Dense(1,activation='sigmoid'))
    opt = Adam(lr=0.0002, beta_1=0.5)
    model.compile(loss='binary_crossentropy',optimizer=opt,metrics=['accuracy'])

    return model



In [4]:
def load_real_samples():
    # load cifar10 dataset
    (trainX,_) , (_,_) = load_data()
    X = trainX.astype('float32')
    # normalize to [-1,1]
    X = (X - 127.5) / 127.5
    return X

def generate_real_samples(dataset,n_samples):
    # choose random instances
    ix = np.random.randint(0,dataset.shape[0],n_samples)
    # retrieve selected images
    X = dataset[ix]
    # generate 'real' class labels (1)
    y = np.ones((n_samples,1))
    return X,y


def generate_fake_samples(n_samples):
    # generate uniform random numbers in [0,1]
    X = np.random.rand(32*32*3*n_samples)
    # update to [-1,1]
    X = -1 + X * 2
    # reshape into a batch of grayscale images
    X = X.reshape((n_samples, 32, 32, 3))
    # generate 'fake' class labels (0)
    y = np.zeros((n_samples, 1))
    return X, y

def train_discriminator(model,dataset,n_iter=20,n_batch=128):
    half_batch = int(n_batch/2)
    for i in range(n_iter):
        # get randomly selected 'real' samples
        X_real,y_real = generate_real_samples(dataset,half_batch)
        # update discriminator on real samples
        _,real_acc = model.train_on_batch(X_real,y_real)
        # generate 'fake' examples
        X_fake,y_fake = generate_fake_samples(half_batch)
        # update discriminator on fake samples
        _,fake_acc = model.train_on_batch(X_fake,y_fake)
        # summarize performance
        print('>%d real=%.0f%% fake=%.0f%%' % (i+1,real_acc*100,fake_acc*100))

def define_generator(latent_dim):
    model = Sequential()

    n_nodes = 256*4*3
    model.add(Dense(n_nodes,input_dim=latent_dim))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Reshape((4,4,256)))
    # upsample to 8x8
    model.add(Conv2DTranspose(128,(4,4),strides=(2,2),padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    # upsample to 16x16
    model.add(Conv2DTranspose(128,(4,4),strides=(2,2),padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    # 
    model.add(Conv2DTranspose(128,(4,4),strides=(2,2),padding='same'))
    model.add(LeakyReLU(alpha=0.2))

    model.add(Conv2D(3,(3,3),activation='tanh',padding='same'))
    return model
def generate_latent_points(latent_dim,n_samples):
    # generate points in the latent space
    x_input = np.random.randn(latent_dim*n_samples)
    # reshape
    x_input = x_input.reshape(n_samples,latent_dim)
    
    return x_input



In [5]:
# define the size of the latent space
latent_dim = 100
# define the generator model
model = define_generator(latent_dim)

model = define_discriminator()
dataset = load_real_samples()
train_discriminator(model,dataset)

c:\Users\fancyma\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\optimizers\optimizer_v2\adam.py:114: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super().__init__(name, **kwargs)


>1 real=47% fake=0%
>2 real=94% fake=8%
>3 real=98% fake=8%
>4 real=100% fake=45%
>5 real=98% fake=69%
>6 real=94% fake=95%
>7 real=98% fake=100%
>8 real=100% fake=100%
>9 real=100% fake=100%
>10 real=98% fake=100%
>11 real=100% fake=100%
>12 real=100% fake=100%
>13 real=98% fake=100%
>14 real=98% fake=100%
>15 real=98% fake=100%
>16 real=100% fake=100%
>17 real=100% fake=100%
>18 real=98% fake=100%
>19 real=100% fake=100%
>20 real=100% fake=100%
